1) SetUp para que todo funcione correctamente

In [35]:
import os
import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix, accuracy_score
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report




PROJECT_ROOT = Path.cwd().parent.parent.parent
sys.path.insert(0, str(PROJECT_ROOT))
from src.common.minio_client import download_df_parquet


print("✓ Setup completado")

✓ Setup completado


2) Extraemos desde MINIO los dos primeros meses de 2025

In [9]:
access_key = os.getenv("MINIO_ACCESS_KEY")
secret_key = os.getenv("MINIO_SECRET_KEY")
if not access_key or not secret_key:
    raise ValueError("MINIO_ACCESS_KEY y MINIO_SECRET_KEY deben estar definidas")

print("Cargando datasets desde MinIO...")

MINIO_PATH = "grupo5/aggregations/DataFrameGroupedByMin=30.parquet"

df = download_df_parquet(access_key, secret_key, MINIO_PATH)

print(f"✓ Total:   {df.shape[0]:,} filas × {df.shape[1]} columnas")

Cargando datasets desde MinIO...
✓ Total:   18,494,599 filas × 80 columnas


In [11]:
df

,stop_id,route_id,direction,merge_time,is_unscheduled_max,is_weekend_max,temp_extreme_max,afecta_previo_max,afecta_durante_max,afecta_despues_max,...,tipo_referente_Parade_sum,tipo_referente_Street Event_sum,tipo_referente_nba_sum,tipo_referente_nhl_sum,tipo_referente_Concierto_sum,tipo_referente_Athletic Race / Tour_sum,tipo_referente_mlb_sum,tipo_referente_baseball/mlb_sum,tipo_referente_basketball/nba_sum,tipo_referente_hockey/nhl_sum
0,101N,1,N,2025-01-01 01:00:00,False,0,1.0,0,0,0,...,0,0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
1,101N,1,N,2025-01-01 02:00:00,False,0,1.0,0,0,0,...,0,0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
2,101N,1,N,2025-01-01 02:30:00,False,0,1.0,0,0,0,...,0,0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
3,101N,1,N,2025-01-01 03:00:00,False,0,0.0,0,0,0,...,0,0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
4,101N,1,N,2025-01-01 03:30:00,False,0,0.0,0,0,0,...,0,0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18494594,S31S,SI,S,2025-12-31 09:30:00,False,0,1.0,0,0,0,...,0,0,NaN,NaN,0.0,NaN,NaN,NaN,NaN,0.0
18494595,S31S,SI,S,2025-12-31 10:00:00,False,0,1.0,0,0,0,...,0,0,NaN,NaN,0.0,NaN,NaN,NaN,NaN,0.0
18494596,S31S,SI,S,2025-12-31 10:30:00,False,0,1.0,0,0,0,...,0,0,NaN,NaN,0.0,NaN,NaN,NaN,NaN,0.0
18494597,S31S,SI,S,2025-12-31 11:00:00,False,0,0.0,0,0,0,...,0,0,NaN,NaN,0.0,NaN,NaN,NaN,NaN,0.0


3) Preparamos el dataframe para trabajar con lo que necesita el modelo

In [16]:
features = [
    "delay_seconds_mean",
    "lagged_delay_1_mean",
    "lagged_delay_2_mean",
    "route_rolling_delay_mean",
    "actual_headway_seconds_mean",
    "seconds_since_last_alert_mean",
    "afecta_previo_max",
    "hour_sin_first",
    "hour_cos_first",
    "dow_first",
    "is_weekend_max",
    "is_unscheduled_max",
    "temp_extreme_max",
    "stops_to_end_mean",
    "scheduled_time_to_end_mean",
    "num_updates_sum",
    "match_key_nunique",
    "direction",
    "route_id",
]

target = "alert_in_next_15m_max"

cols_needed = features + [target]

df_model = df[cols_needed].copy()

print("Shape inicial:", df_model.shape)
print("\nNaN iniciales:")
print(df_model.isna().sum())

Shape inicial: (18494599, 20)

NaN iniciales:
delay_seconds_mean                230442
lagged_delay_1_mean               785037
lagged_delay_2_mean              1460780
route_rolling_delay_mean          221327
actual_headway_seconds_mean       296195
seconds_since_last_alert_mean     166869
afecta_previo_max                      0
hour_sin_first                         0
hour_cos_first                         0
dow_first                              0
is_weekend_max                         0
is_unscheduled_max                     0
temp_extreme_max                   15948
stops_to_end_mean                      0
scheduled_time_to_end_mean             0
num_updates_sum                        0
match_key_nunique                      0
direction                              0
route_id                               0
alert_in_next_15m_max                  0
dtype: int64


4) Limpieza inicial

In [36]:
# Eliminar filas sin target
df_model = df_model.dropna(subset=[target]).copy()
df_model[target] = pd.to_numeric(df_model[target], errors="coerce")
df_model = df_model.dropna(subset=[target]).copy()
df_model[target] = df_model[target].astype(int)

# Separar tipos de columnas
categorical_features = ["direction", "route_id"]
binary_features = [
    "afecta_previo_max",
    "is_weekend_max",
    "is_unscheduled_max",
    "temp_extreme_max",
]
numeric_features = [
    c for c in features
    if c not in categorical_features
]

# Reemplazar inf por NaN solo en numéricas
df_model[numeric_features] = df_model[numeric_features].replace([np.inf, -np.inf], np.nan)

# Asegurar binarias numéricas
for col in binary_features:
    df_model[col] = pd.to_numeric(df_model[col], errors="coerce")

# Categóricas a string
for col in categorical_features:
    df_model[col] = df_model[col].astype("string")

df_model = df_model.dropna(subset=features).copy()

print("Shape tras dropna en features:", df_model.shape)
print(df_model[features + [target]].isna().sum())

Shape tras dropna en features: (16532368, 20)
delay_seconds_mean               0
lagged_delay_1_mean              0
lagged_delay_2_mean              0
route_rolling_delay_mean         0
actual_headway_seconds_mean      0
seconds_since_last_alert_mean    0
afecta_previo_max                0
hour_sin_first                   0
hour_cos_first                   0
dow_first                        0
is_weekend_max                   0
is_unscheduled_max               0
temp_extreme_max                 0
stops_to_end_mean                0
scheduled_time_to_end_mean       0
num_updates_sum                  0
match_key_nunique                0
direction                        0
route_id                         0
alert_in_next_15m_max            0
dtype: int64


5) Comprobación de que todo esta bien antes de empezar a entrenar el modelo

In [37]:
print("Resumen de tipos:")
print(df_model[features + [target]].dtypes)

print("\nDistribución target:")
print(df_model[target].value_counts(dropna=False))
print("\nProporción alertas:", df_model[target].mean())

print("\nNaN en features:")
print(df_model[features].isna().sum().sort_values(ascending=False))

Resumen de tipos:
delay_seconds_mean               float32
lagged_delay_1_mean              float32
lagged_delay_2_mean              float32
route_rolling_delay_mean         float32
actual_headway_seconds_mean      float32
seconds_since_last_alert_mean    float32
afecta_previo_max                  int32
hour_sin_first                   float32
hour_cos_first                   float32
dow_first                          Int32
is_weekend_max                     Int32
is_unscheduled_max                  bool
temp_extreme_max                 float64
stops_to_end_mean                float64
scheduled_time_to_end_mean       float32
num_updates_sum                  float32
match_key_nunique                  int64
direction                         string
route_id                          string
alert_in_next_15m_max              int64
dtype: object

Distribución target:
alert_in_next_15m_max
0    14164141
1     2368227
Name: count, dtype: int64

Proporción alertas: 0.14324790011932956

NaN en f

6) Mas comprobaciones (sobre todo de nulos, ya que no son compatibles):

In [38]:
X = df_model[features].copy()
y = df_model[target].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nNaN en X:")
print(X.isna().sum())

print("\nNaN en y:")
print(y.isna().sum())

X shape: (16532368, 19)
y shape: (16532368,)

NaN en X:
delay_seconds_mean               0
lagged_delay_1_mean              0
lagged_delay_2_mean              0
route_rolling_delay_mean         0
actual_headway_seconds_mean      0
seconds_since_last_alert_mean    0
afecta_previo_max                0
hour_sin_first                   0
hour_cos_first                   0
dow_first                        0
is_weekend_max                   0
is_unscheduled_max               0
temp_extreme_max                 0
stops_to_end_mean                0
scheduled_time_to_end_mean       0
num_updates_sum                  0
match_key_nunique                0
direction                        0
route_id                         0
dtype: int64

NaN en y:
0


8) Split/Train test

In [40]:
split = int(len(df_model) * 0.8)

X_train = X.iloc[:split].copy()
X_test  = X.iloc[split:].copy()

y_train = y.iloc[:split].copy()
y_test  = y.iloc[split:].copy()

print("Train:", X_train.shape, y_train.shape)
print("Test:", X_test.shape, y_test.shape)

Train: (13225894, 19) (13225894,)
Test: (3306474, 19) (3306474,)


9) Ultima comprobación de nulos

In [41]:
print("NaN en X_train:")
print(X_train.isna().sum().sort_values(ascending=False))

print("\nNaN en X_test:")
print(X_test.isna().sum().sort_values(ascending=False))

print("\nNaN en y_train:", y_train.isna().sum())
print("NaN en y_test:", y_test.isna().sum())

num_train = X_train.select_dtypes(include=[np.number])
num_test = X_test.select_dtypes(include=[np.number])

print("\nInf en X_train:")
print(np.isinf(num_train).sum())

print("\nInf en X_test:")
print(np.isinf(num_test).sum())

NaN en X_train:
delay_seconds_mean               0
lagged_delay_1_mean              0
lagged_delay_2_mean              0
route_rolling_delay_mean         0
actual_headway_seconds_mean      0
seconds_since_last_alert_mean    0
afecta_previo_max                0
hour_sin_first                   0
hour_cos_first                   0
dow_first                        0
is_weekend_max                   0
is_unscheduled_max               0
temp_extreme_max                 0
stops_to_end_mean                0
scheduled_time_to_end_mean       0
num_updates_sum                  0
match_key_nunique                0
direction                        0
route_id                         0
dtype: int64

NaN en X_test:
delay_seconds_mean               0
lagged_delay_1_mean              0
lagged_delay_2_mean              0
route_rolling_delay_mean         0
actual_headway_seconds_mean      0
seconds_since_last_alert_mean    0
afecta_previo_max                0
hour_sin_first                   0
hour_cos_f

10) Creación del pipeline

In [42]:
categorical_features = ["direction", "route_id"]
numeric_features = [c for c in features if c not in categorical_features]

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(
        max_iter=300,
        class_weight="balanced",
        random_state=42,
        solver="lbfgs"
    ))
])

pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers co

11) Entrenamiento del modelo

In [43]:

y_prob = pipeline.predict_proba(X_test)[:, 1]
y_pred = (y_prob >= 0.5).astype(int)

print("F1:", f1_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, zero_division=0))
print("Recall:", recall_score(y_test, y_pred, zero_division=0))

print("\nClassification report:")
print(classification_report(y_test, y_pred, zero_division=0))

print("\nConfusion matrix:")
print(confusion_matrix(y_test, y_pred))

F1: 0.3688513962579287
Precision: 0.2421135868654716
Recall: 0.7740267196832715

Classification report:
              precision    recall  f1-score   support

           0       0.94      0.60      0.73   2839702
           1       0.24      0.77      0.37    466772

    accuracy                           0.63   3306474
   macro avg       0.59      0.69      0.55   3306474
weighted avg       0.84      0.63      0.68   3306474


Confusion matrix:
[[1708746 1130956]
 [ 105478  361294]]


12) Evaluacion con otros umbrales

In [44]:
for thr in [0.5, 0.3, 0.2, 0.1, 0.05]:
    y_pred_thr = (y_prob >= thr).astype(int)
    print(f"\nUmbral = {thr}")
    print(classification_report(y_test, y_pred_thr, zero_division=0))


Umbral = 0.5
              precision    recall  f1-score   support

           0       0.94      0.60      0.73   2839702
           1       0.24      0.77      0.37    466772

    accuracy                           0.63   3306474
   macro avg       0.59      0.69      0.55   3306474
weighted avg       0.84      0.63      0.68   3306474


Umbral = 0.3
              precision    recall  f1-score   support

           0       0.97      0.27      0.42   2839702
           1       0.18      0.95      0.30    466772

    accuracy                           0.37   3306474
   macro avg       0.57      0.61      0.36   3306474
weighted avg       0.86      0.37      0.41   3306474


Umbral = 0.2
              precision    recall  f1-score   support

           0       0.97      0.17      0.29   2839702
           1       0.16      0.97      0.28    466772

    accuracy                           0.28   3306474
   macro avg       0.57      0.57      0.28   3306474
weighted avg       0.86      0.2

14) Probabilidades para comprobar

In [45]:
print("Probabilidades:")
print("min:", y_prob.min())
print("max:", y_prob.max())
print("mean:", y_prob.mean())

Probabilidades:
min: 5.695361906666512e-32
max: 0.9969042305299293
mean: 0.43671393869451264


In [47]:
import optuna

categorical_features = ["direction", "route_id"]
numeric_features = [c for c in features if c not in categorical_features]

# split temporal interno sobre train
split_val = int(len(X_train) * 0.8)

X_tr = X_train.iloc[:split_val].copy()
y_tr = y_train.iloc[:split_val].copy()

X_val = X_train.iloc[split_val:].copy()
y_val = y_train.iloc[split_val:].copy()

def objective(trial):
    C = trial.suggest_float("C", 1e-3, 10.0, log=True)
    class_weight = trial.suggest_categorical("class_weight", [None, "balanced"])
    threshold = trial.suggest_float("threshold", 0.1, 0.9)

    numeric_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_features),
            ("cat", categorical_transformer, categorical_features),
        ]
    )

    pipeline_trial = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(
            C=C,
            class_weight=class_weight,
            max_iter=300,
            random_state=42,
            solver="lbfgs"
        ))
    ])

    pipeline_trial.fit(X_tr, y_tr)

    y_val_prob = pipeline_trial.predict_proba(X_val)[:, 1]
    y_val_pred = (y_val_prob >= threshold).astype(int)

    return f1_score(y_val, y_val_pred, zero_division=0)

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=30)

print("Mejor F1:", study.best_value)
print("Mejores parámetros:", study.best_params)

[I 2026-03-25 16:17:54,646] A new study created in memory with name: no-name-2f903953-2fbf-48df-99c5-274469d99893
[I 2026-03-25 16:19:11,286] Trial 0 finished with value: 0.009385884109948927 and parameters: {'C': 0.0012292079840520507, 'class_weight': None, 'threshold': 0.4407695309222861}. Best is trial 0 with value: 0.009385884109948927.
[I 2026-03-25 16:20:35,144] Trial 1 finished with value: 0.2774920620723581 and parameters: {'C': 0.0013819899281380323, 'class_weight': 'balanced', 'threshold': 0.6774971924094773}. Best is trial 1 with value: 0.2774920620723581.
[I 2026-03-25 16:22:03,975] Trial 2 finished with value: 0.39437921963821587 and parameters: {'C': 0.4772159177052329, 'class_weight': None, 'threshold': 0.22220217958308786}. Best is trial 2 with value: 0.39437921963821587.
[I 2026-03-25 16:25:51,195] Trial 3 finished with value: 1.653667006587107e-05 and parameters: {'C': 0.0032504542861501074, 'class_weight': None, 'threshold': 0.6754837996145223}. Best is trial 2 with 

Mejor F1: 0.39601509154456865
Mejores parámetros: {'C': 0.10284240577738168, 'class_weight': None, 'threshold': 0.2071708939528965}


In [48]:
best_params = study.best_params

best_C = best_params["C"]
best_class_weight = best_params["class_weight"]
best_threshold = best_params["threshold"]

categorical_features = ["direction", "route_id"]
numeric_features = [c for c in features if c not in categorical_features]

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

final_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(
        C=best_C,
        class_weight=best_class_weight,
        max_iter=300,
        random_state=42,
        solver="lbfgs"
    ))
])

final_pipeline.fit(X_train, y_train)

y_test_prob = final_pipeline.predict_proba(X_test)[:, 1]
y_test_pred = (y_test_prob >= best_threshold).astype(int)

from sklearn.metrics import f1_score, precision_score, recall_score, classification_report, confusion_matrix

print("F1 test:", f1_score(y_test, y_test_pred, zero_division=0))
print("Precision test:", precision_score(y_test, y_test_pred, zero_division=0))
print("Recall test:", recall_score(y_test, y_test_pred, zero_division=0))

print("\nClassification report:")
print(classification_report(y_test, y_test_pred, zero_division=0))

print("\nConfusion matrix:")
print(confusion_matrix(y_test, y_test_pred))

F1 test: 0.39788032510593435
Precision test: 0.30820679065005974
Recall test: 0.5611476266785497

Classification report:
              precision    recall  f1-score   support

           0       0.92      0.79      0.85   2839702
           1       0.31      0.56      0.40    466772

    accuracy                           0.76   3306474
   macro avg       0.61      0.68      0.62   3306474
weighted avg       0.83      0.76      0.79   3306474


Confusion matrix:
[[2251785  587917]
 [ 204844  261928]]
